# TSA_ch12_lag_windows

**Published in:** Time Series Analysis  
**Author:** Daniel Traian Pele  
**Description:** Lag-window (Blackman-Tukey) spectral estimation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

COLORS = {'blue':'#1A3A6E','red':'#DC3545','green':'#2E7D32','amber':'#B5853F','orange':'#E6802E','purple':'#8E44AD','gray':'#666666'}

In [ ]:
M = 64  # truncation point
k = np.arange(-M, M+1)

def bartlett(k, M):   return 1 - np.abs(k)/M
def tukey_hanning(k, M): return 0.5*(1 + np.cos(np.pi*k/M))
def parzen(k, M):
    u = np.abs(k) / M
    w = np.where(u <= 0.5,
                 1 - 6*u**2 + 6*u**3,
                 2*(1-u)**3)
    return w

windows = {
    'Bartlett':      (bartlett(k, M),      COLORS['blue']),
    'Tukey-Hanning': (tukey_hanning(k, M), COLORS['orange']),
    'Parzen':        (parzen(k, M),        COLORS['red']),
}

# Spectral windows via FFT
omega = np.linspace(-np.pi, np.pi, 2048)

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for (name, (w, col)) in windows.items():
    axes[0, 0].plot(k, w, color=col, lw=1.8, label=name)
axes[0, 0].set_title('Lag Windows')
axes[0, 0].set_xlabel('Lag k')
axes[0, 0].set_ylabel('w(k)')

for i, (name, (w, col)) in enumerate(windows.items()):
    W = np.abs(np.fft.fftshift(np.fft.fft(w, n=2048)))
    axes[0, 1].plot(omega, W / W.max(), color=col, lw=1.8, label=name)
axes[0, 1].set_title('Spectral Windows (normalised)')
axes[0, 1].set_xlabel('Frequency ω')
axes[0, 1].set_ylabel('|W(ω)|')
axes[0, 1].set_xlim(-np.pi/4, np.pi/4)

# Log spectral windows
for name, (w, col) in windows.items():
    W = np.abs(np.fft.fftshift(np.fft.fft(w, n=2048)))
    W = np.maximum(W, 1e-10)
    axes[1, 0].plot(omega, 20*np.log10(W/W.max()), color=col, lw=1.8, label=name)
axes[1, 0].set_title('Spectral Windows (dB)')
axes[1, 0].set_xlabel('Frequency ω')
axes[1, 0].set_ylabel('dB')
axes[1, 0].set_ylim(-80, 5)
axes[1, 0].set_xlim(-np.pi/4, np.pi/4)

# Example: ACVF of AR(1) * lag windows
phi = 0.6
max_lag = M
lags = np.arange(-max_lag, max_lag+1)
acvf = phi**np.abs(lags)
for name, (w, col) in windows.items():
    axes[1, 1].plot(lags, acvf * w, color=col, lw=1.8, label=name)
axes[1, 1].set_title('ACVF × Lag Window (AR(1) φ=0.6)')
axes[1, 1].set_xlabel('Lag k')
axes[1, 1].set_ylabel('γ(k)·w(k)')

for ax in axes.flat:
    ax.set_facecolor('none')
    ax.spines[['top','right']].set_visible(False)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=3, frameon=False)
fig.patch.set_alpha(0)
plt.tight_layout()
plt.savefig('lag_windows.pdf', bbox_inches='tight')
plt.show()